In [1]:
import pandas as pd

# 1) Adding ticket uuid and attachment id to the current dataframe

In [2]:
raw_data = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv")

In [3]:
raw_data

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,attachment_id,s3_link
0,4070071 967072\nAntrag auf Erlass eines Pfändu...,01.02.2022/1643724050_Doc_01022022_000000001_1...,attachment_and_transfer_order,4070071 967072\nantrag auf erlass eines pfändu...,NaN,False,False,NaN,NaN
1,Antrag auf Erlass eines Pfänd 070071 967072\nÜ...,01.02.2022/1643724051_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfänd 070071 967072\nü...,NaN,False,False,NaN,NaN
2,Antrag auf Erlass eines Pfändun\n4070071 96707...,01.02.2022/1643724052_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändun\n4070071 96707...,NaN,False,False,NaN,NaN
3,Antrag auf Erlass eines Pfändungs\n4 070071 96...,01.02.2022/1643724052_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändungs\n4 070071 96...,NaN,False,False,NaN,NaN
4,Antrag auf Erlass eines Pfändu\n4070071 967072...,01.02.2022/1643724053_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändu\n4070071 967072...,NaN,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...
5350,"Myriam Wehmeyer\nc/o Amtsgericht Rotenburg,Am\...",data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,"myriam wehmeyer\nc/o amtsgericht rotenburg,am\...",NaN,False,False,NaN,NaN
5351,"Paul-Rohland-Str.2\n06712 Zeitz, Elster\nTel. ...",data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,"paul-rohland-str.2\n06712 zeitz, elster\ntel. ...",NaN,False,False,NaN,NaN
5352,Wolfgang Lurz\nSchragenhofstraße 27\n80992 Mün...,data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,wolfgang lurz\nschragenhofstraße 27\n80992 mün...,NaN,False,False,NaN,NaN
5353,Obergerichtsvollzieher\n33104 Paderborn\nWagen...,data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,obergerichtsvollzieher\n33104 paderborn\nwagen...,NaN,False,False,NaN,NaN


In [4]:
attachment_ids = raw_data[~raw_data['attachment_id'].isna()]
attachment_ids

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,attachment_id,s3_link
5030,ANNETTE ZIMMER\nDie nachstehend gewählten Form...,NaN,fp_protocol,annette zimmer\ndie nachstehend gewählten form...,NaN,NaN,False,23973761582236-1,s3://pair-data-engineering-new/ocr_prepared_ou...
5031,Sven Wiegand\nDie nachstehend gewählten Formul...,NaN,fp_protocol,sven wiegand\ndie nachstehend gewählten formul...,NaN,NaN,False,24693116189212-1,s3://pair-data-engineering-new/ocr_prepared_ou...
5032,JÖRG SCHMIDT\nDie nachstehend gewählten Formul...,NaN,fp_protocol,jörg schmidt\ndie nachstehend gewählten formul...,NaN,NaN,False,24823927696284-3,s3://pair-data-engineering-new/ocr_prepared_ou...
5033,Sven Wiegand\nDie nachstehend gewählten Formul...,NaN,fp_protocol,sven wiegand\ndie nachstehend gewählten formul...,NaN,NaN,False,24824722387996-2,s3://pair-data-engineering-new/ocr_prepared_ou...
5034,Sven Wiegand\nDie nachstehend gewählten Formul...,NaN,fp_protocol,sven wiegand\ndie nachstehend gewählten formul...,NaN,NaN,False,24825525155740-3,s3://pair-data-engineering-new/ocr_prepared_ou...
...,...,...,...,...,...,...,...,...,...
5244,STEFAN HOFFMANN\nDie nachstehend gewählten For...,NaN,fp_protocol,stefan hoffmann\ndie nachstehend gewählten for...,NaN,NaN,False,49736954,s3://pair-data-engineering-new/ocr_prepared_ou...
5245,Manuel Stember\nDie nachstehend gewählten Form...,NaN,fp_protocol,manuel stember\ndie nachstehend gewählten form...,NaN,NaN,False,49747921,s3://pair-data-engineering-new/ocr_prepared_ou...
5246,S. Schmitz\nDie nachstehend gewählten Formulie...,NaN,fp_protocol,s. schmitz\ndie nachstehend gewählten formulie...,NaN,NaN,False,49775489,s3://pair-data-engineering-new/ocr_prepared_ou...
5247,Frank Lüers\nDie nachstehend gewählten Formuli...,NaN,fp_protocol,frank lüers\ndie nachstehend gewählten formuli...,NaN,NaN,False,49778889,s3://pair-data-engineering-new/ocr_prepared_ou...


This 219 fp protocol data is already in our textract_jobs table, only add them to the llm_ticket_labels. Get theirs ticket uuids from the llm_tickets table

In [5]:
attachment_ids_to_fetch_ticket_uuid = attachment_ids['attachment_id'].tolist()
len(attachment_ids_to_fetch_ticket_uuid)

219

In [6]:
attachment_ids['attachment_id'].nunique()

218

In [7]:
attachment_ids['attachment_id'].isna().sum()

np.int64(0)

In [8]:
from python_utilities.db_connection import DbConnection
analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')
a_id_to_t_id = analytics_db.sql_to_df(f"""
    SELECT la.attachment_id, lt.ticket_uuid
    FROM llm_attachments la LEFT JOIN llm_tickets lt ON la.ticket_uuid = lt.ticket_uuid
    WHERE la.attachment_id IN ({','.join([str(id) for id in attachment_ids_to_fetch_ticket_uuid])})
""")


INFO [2026-03-02 11:25:29] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


In [9]:
a_id_to_t_id

,attachment_id,ticket_uuid
0,44529743,4fe9b711-1c87-5072-8cee-30ef653866b7
1,44531561,0d29c484-086e-52c5-8e2d-1225798e8921
2,44689202,606b4c67-602f-5eb1-b278-32e464a5ebbe
3,44707482,92cfa0c5-cc22-5830-af7a-526d17fab297
4,44708172,22f3f9bd-48dd-50fb-aab7-9396d9cc4db2
...,...,...
207,49736954,15dae806-8d02-518a-bf32-2c51fe3e4792
208,49747921,None
209,49775489,5bebc9d4-4923-557a-a3e0-ae948c1396b2
210,49778889,ffa1e6d2-908a-54d7-ab14-d6d6c674bc33


In [10]:
# set ticket uuids in raw data for corresponding attachment ids
raw_data_with_ticket_uuids = raw_data.merge(a_id_to_t_id, on='attachment_id', how='left')

In [11]:
raw_data_with_ticket_uuids.isna().sum()

text                0
object_key       1314
document_type       0
cleaned_text        0
data             4888
is_pfub           308
is_ladung           0
attachment_id    5136
s3_link          5136
ticket_uuid      5164
dtype: int64

In [ ]:
len(raw_data_with_ticket_uuids) - raw_data_with_ticket_uuids.isna().sum() # number of NOT NA

text             5355
object_key       4041
document_type    5355
cleaned_text     5355
data              467
is_pfub          5047
is_ladung        5355
attachment_id     219
s3_link           219
ticket_uuid       191
dtype: int64

In [13]:
raw_data_with_ticket_uuids

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,attachment_id,s3_link,ticket_uuid
0,4070071 967072\nAntrag auf Erlass eines Pfändu...,01.02.2022/1643724050_Doc_01022022_000000001_1...,attachment_and_transfer_order,4070071 967072\nantrag auf erlass eines pfändu...,NaN,False,False,NaN,NaN,NaN
1,Antrag auf Erlass eines Pfänd 070071 967072\nÜ...,01.02.2022/1643724051_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfänd 070071 967072\nü...,NaN,False,False,NaN,NaN,NaN
2,Antrag auf Erlass eines Pfändun\n4070071 96707...,01.02.2022/1643724052_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändun\n4070071 96707...,NaN,False,False,NaN,NaN,NaN
3,Antrag auf Erlass eines Pfändungs\n4 070071 96...,01.02.2022/1643724052_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändungs\n4 070071 96...,NaN,False,False,NaN,NaN,NaN
4,Antrag auf Erlass eines Pfändu\n4070071 967072...,01.02.2022/1643724053_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändu\n4070071 967072...,NaN,False,False,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
5350,"Myriam Wehmeyer\nc/o Amtsgericht Rotenburg,Am\...",data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,"myriam wehmeyer\nc/o amtsgericht rotenburg,am\...",NaN,False,False,NaN,NaN,NaN
5351,"Paul-Rohland-Str.2\n06712 Zeitz, Elster\nTel. ...",data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,"paul-rohland-str.2\n06712 zeitz, elster\ntel. ...",NaN,False,False,NaN,NaN,NaN
5352,Wolfgang Lurz\nSchragenhofstraße 27\n80992 Mün...,data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,wolfgang lurz\nschragenhofstraße 27\n80992 mün...,NaN,False,False,NaN,NaN,NaN
5353,Obergerichtsvollzieher\n33104 Paderborn\nWagen...,data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,obergerichtsvollzieher\n33104 paderborn\nwagen...,NaN,False,False,NaN,NaN,NaN


In [14]:
import uuid
import hashlib


def generate_uuid(text: str, purpose: str) -> str:
    """
    Generate a UUID based on the input text.

    Args:
        text (str): The input text to generate the UUID from.
        purpose (str): The purpose for which the UUID is being generated.


    Returns:
        str: A UUID string generated from the input text and purpose.
    """
    text = text[:100]  # Limit the text to the first 100 characters
    signature = f"{purpose}:{text}"
    # Use nil UUID as namespace (compatible with all Python versions)
    nil_namespace = uuid.UUID('00000000-0000-0000-0000-000000000000')
    return str(uuid.uuid5(nil_namespace, signature))



def generate_hash_from_text(text: str) -> str:
    text = text[:100]  # Limit the text to the first 100 characters
    """Generate SHA-256 hash from text string"""
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def extract_filename_from_object_key(object_key: str) -> str:
    if pd.isna(object_key):
        return None
    return object_key.split('/')[-1]

def extract_s3_key(row):
    created_at = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    base = "ocr_source_files"
    date_folder = created_at.split(' ')[0]  # Extract date part: YYYY-MM-DD
    egvp_id = row["egvp_id"]
    egvp_folder = f"egvp_id_{egvp_id}"
    full_path = f"{base}/{date_folder}/{egvp_folder}/{row['file_name']}"
    return full_path

add ticket_uuid

In [20]:
raw_data_with_ticket_uuids['ticket_uuid'] = raw_data_with_ticket_uuids.apply(lambda row: generate_uuid(row['text'], purpose="ticket_uuid") if pd.isna(row['ticket_uuid']) == True else row['ticket_uuid'], axis=1)

In [21]:
raw_data_with_ticket_uuids['ticket_uuid'].isna().sum()

np.int64(0)

add attachment_id

In [22]:
raw_data_with_ticket_uuids['attachment_id'] = raw_data_with_ticket_uuids.apply(lambda row: generate_uuid(row['text'], purpose="attachment_id") if pd.isna(row['attachment_id']) == True else row['attachment_id'], axis=1)

In [23]:
raw_data_with_ticket_uuids['attachment_id'].isna().sum()

np.int64(0)

add textract related fields

In [24]:
raw_data_with_ticket_uuids['textract_job_id'] = raw_data_with_ticket_uuids['text'].apply(lambda x: generate_hash_from_text(x))
raw_data_with_ticket_uuids['textract_s3_link'] = raw_data_with_ticket_uuids['textract_job_id'].apply(lambda x: f"s3://pair-data-engineering-new/ocr_prepared_output/{x}.json")

In [26]:
raw_data_with_ticket_uuids['textract_job_id'].isna().sum(), raw_data_with_ticket_uuids['textract_s3_link'].isna().sum()

(np.int64(0), np.int64(0))

In [28]:
new_column_order = ['ticket_uuid', 'attachment_id'] + [col for col in raw_data_with_ticket_uuids.columns if col not in ['ticket_uuid', 'attachment_id']]
raw_data_with_ticket_uuids = raw_data_with_ticket_uuids[new_column_order]

In [29]:
raw_data_with_ticket_uuids

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link
0,937b5e2e-f3b3-56bd-9225-f67840039ee1,8e6a76c1-34ed-5fe0-9144-bbd00e1cddd8,4070071 967072\nAntrag auf Erlass eines Pfändu...,01.02.2022/1643724050_Doc_01022022_000000001_1...,attachment_and_transfer_order,4070071 967072\nantrag auf erlass eines pfändu...,NaN,False,False,NaN,aa53a00a4be3004cf77a29bad626bfd877a85217942166...,s3://pair-data-engineering-new/ocr_prepared_ou...
1,7e7ac68d-b2df-50d2-963d-200aa44b036b,417ecde8-b807-5de4-b066-f70f811c8f6a,Antrag auf Erlass eines Pfänd 070071 967072\nÜ...,01.02.2022/1643724051_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfänd 070071 967072\nü...,NaN,False,False,NaN,a526cbb704ee1ce316ef92311493578034721ce8d1ac27...,s3://pair-data-engineering-new/ocr_prepared_ou...
2,2d315b63-cb84-56ef-bb02-af6c13dff49b,acc1f4fa-6891-58d3-a3d1-d3f3e54d93ba,Antrag auf Erlass eines Pfändun\n4070071 96707...,01.02.2022/1643724052_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändun\n4070071 96707...,NaN,False,False,NaN,119c66b09111a9c414741acb6a520aceac59eb2c31deeb...,s3://pair-data-engineering-new/ocr_prepared_ou...
3,dc9e9a0c-12a6-5b22-b6c0-52f1647505a2,cfc4d684-af88-506e-8fdb-39984cf93640,Antrag auf Erlass eines Pfändungs\n4 070071 96...,01.02.2022/1643724052_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändungs\n4 070071 96...,NaN,False,False,NaN,e48ca3ce0673313350c06532a2dea4fe82acf6243da3f1...,s3://pair-data-engineering-new/ocr_prepared_ou...
4,6abb8d66-912f-55bb-9b05-50dbd5c6a6f7,ccf92577-4ae3-505d-97eb-25c49e6fbc12,Antrag auf Erlass eines Pfändu\n4070071 967072...,01.02.2022/1643724053_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändu\n4070071 967072...,NaN,False,False,NaN,cb548acfe61af12fd9b1dd59419b673b34b7e32ff0bc1f...,s3://pair-data-engineering-new/ocr_prepared_ou...
...,...,...,...,...,...,...,...,...,...,...,...,...
5350,2bd93acd-08cd-5d66-9d0e-c72b3e1da733,0bd71681-f525-5f65-83eb-952e605409f3,"Myriam Wehmeyer\nc/o Amtsgericht Rotenburg,Am\...",data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,"myriam wehmeyer\nc/o amtsgericht rotenburg,am\...",NaN,False,False,NaN,ea578e5a3589fc0c5b640d644d57e917f384c0458f08ef...,s3://pair-data-engineering-new/ocr_prepared_ou...
5351,079a8404-edb0-5660-8722-8d46098ae215,8e8ef642-2c7a-5bdb-810b-48344c2a235a,"Paul-Rohland-Str.2\n06712 Zeitz, Elster\nTel. ...",data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,"paul-rohland-str.2\n06712 zeitz, elster\ntel. ...",NaN,False,False,NaN,dd4d8ce69f0e90db975a425b96b6c6515f56e912df606c...,s3://pair-data-engineering-new/ocr_prepared_ou...
5352,3321bf02-b3dd-592e-b0c6-e129b3719312,d47fad4c-772a-5f89-80c0-5ac9638b9e7e,Wolfgang Lurz\nSchragenhofstraße 27\n80992 Mün...,data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,wolfgang lurz\nschragenhofstraße 27\n80992 mün...,NaN,False,False,NaN,ce565a09de5e250d5545b02d2a482a08806b1df497e3cd...,s3://pair-data-engineering-new/ocr_prepared_ou...
5353,f5bdf478-897e-5da6-8669-f835dfcb9a38,ec52a8de-3bf8-5660-8501-414aebade679,Obergerichtsvollzieher\n33104 Paderborn\nWagen...,data/aftercourt/ladung_fp_instplan_with_bailif...,bailiff_ip,obergerichtsvollzieher\n33104 paderborn\nwagen...,NaN,False,False,NaN,3aa213ac35184d2007d9386ad4f8b0f5b502ab1cb36b7d...,s3://pair-data-engineering-new/ocr_prepared_ou...


save raw df with new columns, update the data with dvc

In [ ]:
# raw_data_with_ticket_uuids.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv", index=False)